# Team Challenge · Sprint 03-04 - Catálogo de películas

**Dataset:**  
- [movies.csv](./data/movies.csv)
- [ratings.csv](./data/ratings.csv)
- [tags.csv](./data/tags.csv)
- [links.csv](./data/links.csv) 

**Entregables:** Repositorio de GitHub con el código fuente. Puede ser scripts de python o notebook ordenado, código reproducible, README con cómo ejecutar el proyecto.

**Control de versiones:** gestión del proyecto con **GitHub desde el primer día**. Trabajar en equipo en paralelo con **ramas**, **pull requests** y revisión entre compañeros.

**Parte 1 (Pandas):** Data analytics (Pandas)

**Parte 2 (TMDB):** una petición `GET /3/movie/{id}` por película; definid `TMDB_API_KEY` o `TMDB_READ_ACCESS_TOKEN` en el entorno (sin subir claves al repo).

**Parte 3 (Gemini):** añadir `overview_es` a `movies10` a partir de los `overview` de TMDB; definid `GEMINI_API_KEY` o usad `getpass` (sin subir claves al repo). Requiere `pip install google-genai`.

**Parte 4 (opcional):**
- **A)** catálogo ampliado (`pitch_es`, `edad_sugerida`, `temas`).
- **B)** recomendador por género.

## Trabajo en equipo y control de versiones

- Crear el **repositorio en GitHub** al inicio (día 1–2), no al final.
- Definir una rama principal (`main`) y una para desarrollo (`develop`) y ramas por tarea (p. ej. `feature/parte1-pandas`, `feature/tmdb`, `feature/gemini`).
- Integrar el trabajo con **pull requests** hacia `develop`; al menos **una PR revisada y mergeada por miembro** del equipo.
- Evitar trabajar en en `main` es la rama de "producción"
- Resolver conflictos en ramas antes del merge.
- **README:** cómo clonar el repo, instalar dependencias, configurar claves (`TMDB_*`, `GEMINI_API_KEY`) y ejecutar el notebook o scripts.
- **No subir claves** al repositorio (usar `.gitignore` para `.env` si aplica).

## Parte 1: Data analytics (Pandas)

### 1. Ingesta de datos

- Los CSV están en **`Team_Challenges/TC_01_Sprint_03_04/data/`**.
- Cargar con **Pandas** los cuatro ficheros: `movies.csv`, `ratings.csv`, `tags.csv`, `links.csv`.
- Comprobar para cada `DataFrame`: `shape`, columnas, `dtypes`, `head` y conteo de nulos.

In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
# --- Apartado 1: Ingesta de datos ---

In [ ]:
# Ingresamos los datos de los CSV
movies_df = pd.read_csv('data/movies.csv')
ratings_df = pd.read_csv("data/ratings.csv")
tags_df = pd.read_csv("data/tags.csv")
links_df = pd.read_csv("data/links.csv")

In [ ]:
# Comprobar de cada DataFrame: shape, columnas, dytpes, head --> esto lo hacemos con df.info()
movies_df.info()

In [ ]:
ratings_df.info()

In [ ]:
tags_df.info()

In [ ]:
links_df.info()

In [ ]:
# Conteo de nulos. Calculo de nulos del total del DataFrame y %
# Conteo
print(f"Total nulos en movies: {movies_df.isna().sum().sum()}")
print(f"Total nulos en ratings: {ratings_df.isna().sum().sum()}")
print(f"Total nulos en tags: {tags_df.isna().sum().sum()}")
print(f"Total nulos en links: {links_df.isna().sum().sum()}")

In [ ]:
# porcentaje de nulos
print(f"Porcentaje de nulos en movies:\n{round(movies_df.isna().sum() / movies_df.shape[0] * 100, 2)}")
print(f"Porcentaje de nulos en rating:\n{round(ratings_df.isna().sum() / ratings_df.shape[0] * 100, 2)}")
print(f"Porcentaje de nulos en tags:\n{round(tags_df.isna().sum() / tags_df.shape[0] * 100, 2)}")
print(f"Porcentaje de nulos en links:\n{round(links_df.isna().sum() / links_df.shape[0] * 100, 2)}")

Al haber una columna `timestamp` en los DataFrames `ratings_df` y `tags_df` se añade en cada DataFrame la columna `date`. Se normaliza `timestamp` a un formato de fecha más manejable.

In [ ]:
# Añadir columna date para manejar fechas timestamp
ratings_df["date"] = pd.to_datetime(ratings_df["timestamp"], unit='s').dt.date
tags_df["date"] = pd.to_datetime(tags_df["timestamp"], unit='s').dt.date

### 2. Columna `year` desde el título

- En el dataset de películas. el año de estreno suele aparecer **entre paréntesis al final** de `title`, p. ej. `Batman (1989)`.
- Implementad una función (`year_from_title`) que devuelva un entero de cuatro cifras o valor ausente (`NaN` / `<NA>`) si el título no sigue ese patrón.
- Añadid la columna **`year`** a `movies` como numérico (`pd.to_numeric(..., errors="coerce")`).
- Editar el campo `title` para que no contenga el año de estreno.
- Contad cuántas películas **no** tienen año reconocible y mostrad **una pequeña muestra** de sus títulos (casos límite).

In [ ]:
# --- Apartado 2: columna `year` ---
#Funcion:
def year_from_title(title: str):
    if not isinstance(title, str):
        return pd.NA
    partes = title.strip().rsplit('(', 1)
    if len(partes) == 2:
        candidato = partes[1].rstrip(')')
        if len(candidato) == 4 and candidato.isdigit():
            return int(candidato)
    return pd.NA

# Extraer el año
movies_df['year'] = movies_df['title'].apply(year_from_title)
movies_df['year'] = pd.to_numeric(movies_df['year'], errors="coerce").astype("Int64")

# Limpiar el título
movies_df['title'] = movies_df['title'].str.split('(').str[0].str.strip()

# Conteo de peliculas sin año
print(f'Hay {movies_df["year"].isna().sum()} películas sin año.')
movies_df

### 2. Unificación de datos (*merge* / *join*)

- Entender las **claves** entre tablas (p. ej. `movieId` une `movies`, `ratings`, `tags` y `links`).
- Construir un esquema unificado: al menos una tabla **película–usuario–rating** (ratings enriquecida con título y géneros) y otra con **película–tags** si procede.
- Usar `merge` (u operaciones equivalentes) con criterio claro: tipo de unión (*inner* / *left*), duplicados generados y cómo se resuelven.
- Imprimir las 10 primeras filas de las tablas resultantes.
- Dejar documentado qué filas se pierden o se multiplican al unir y **por qué** es aceptable en vuestro caso de uso.

In [ ]:
#Creación tabla pelicula-usuario-rating
ratings_movies_df = ratings_df.merge(movies_df, on='movieId', how='left')
print(f"Nulos del DataFrame ratings_movies: {ratings_movies_df['title'].isna().sum()}")
print(f"Tamaño del DataFrame ratings: {ratings_df.shape}")
print(f"Tamaño del DataFrame ratings_movies: {ratings_movies_df.shape}")
print()

#CREACION TABLA PELICULA-TAGS
tags_movies_df = tags_df.merge(movies_df, on='movieId', how='left')
print(f"Nulos del DataFrame tags_movies: {tags_movies_df['title'].isna().sum()}")
print(f"Tamaño del DataFrame tags: {tags_df.shape}")
print(f"Tamaño del DataFrame tags_movies: {tags_movies_df.shape}")

In [ ]:
print("Primeras filas del DataFrame ratings_movies.")
ratings_movies_df.head(10)

In [ ]:
print("Primeras filas del DataFrame tags_movies.")
tags_movies_df.head(10)

### 4. Agregaciones y segmentación

- Usar `groupby` (por usuario, por película o por género) con al menos una agregación multi-columna (`agg`).
- Comparar dos segmentos (p. ej. usuarios con muchas valoraciones vs. pocos; o por **década** usando la columna `year` del apartado 3) con tablas resumen.

In [ ]:
# Varias agregaciones: media, mediana, recuento, minimo, máximo y cantidad de usuarios
# por cada película
ratings_agg_df = ratings_movies_df.groupby(['movieId', 'title'], as_index=False).agg( # nombre_col=(columna, operacion)
    rating_mean=('rating', 'mean'),
    rating_median=('rating', 'median'),
    rating_count=('rating', 'count'),
    rating_min=('rating', 'min'),
    rating_max=('rating', 'max'),
    user_count=('userId', 'nunique')
)

# Redondeo rating_mean
ratings_agg_df['rating_mean'] = ratings_agg_df['rating_mean'].round(2)

ratings_agg_df.head()

Hemos decidido segmentar por cantidad de valoraciones que tiene cada película haciendo 3 bloques:

* **Pocas valoraciones**: hasta 30 valoraciones.
* **Valoraciones medias**: entre 31 y 50 valoraciones.
* **Muchas valoraciones**: más de 50 valoraciones.

Una vez establecida la segmentación:

1. `peliculas`: cuántas películas hay en cada segmento.
2. `rating_mean_s`: media de las puntuaciones por segmento.
3. `valoraciones_totales_s`: cantidad de valoraciones por segmento.
4. `user_count_s`: cantidad de usuarios que hace valoración por segmento.


In [ ]:
# Creación de segmentos
ratings_agg_df['segmento_rating'] = pd.cut(
    ratings_agg_df['rating_count'],
    bins=[0, 30, 50, float('inf')],
    labels=[
        "Pocas valoraciones",
        "Valoraciones medias",
        "Muchas valoraciones"
    ]
)

# Agregaciones
segmentos_agg_df = ratings_agg_df.groupby('segmento_rating', as_index=False, observed=True).agg(
    peliculas=('movieId', 'count'),
    rating_mean_s=('rating_mean', 'mean'), 
    valoraciones_totales_s=('rating_count', 'sum'),
    user_total_s=('user_count', 'sum')
)

# Redondeo en columna rating_mean_s
segmentos_agg_df['rating_mean_s'] = segmentos_agg_df['rating_mean_s'].round(2) 

segmentos_agg_df

Para resolver las diferentes agregaciones según la década partimos del `DataFrame` `ratings_agg_df`.  
Se añade la columna `decade` con los años por década: 1950, 1960...

Se sacan las siguientes agregaciones:

1. `peliculas`: cantidad de películas por década.
2. `rating_mean_d`: puntuación media por década.
3. `valoraciones_totales_d`: cantidad de valoraciones por década.
4. `user_count_d`: cantidad de usuarios que han hecho una valoración por década.


In [ ]:
ratings_agg_df['decade'] = ratings_movies_df['year'] // 10 * 10

# Agregaciones
decada_df = ratings_agg_df.dropna(subset=['decade']).groupby('decade', as_index=False).agg(
    peliculas=('movieId', 'count'), 
    rating_mean_d=('rating_mean', 'mean'),
    valoraciones_totales_d=('rating_count', 'sum'),
    user_total_s=('user_count', 'sum')
)

# Redondeo en la solumna rating_medio_df
decada_df['rating_mean_d'] = decada_df['rating_mean_d'].round(2)
decada_df

### 5. Preguntas sobre los datos (`movies`)

**Requisito:** haber creado la columna `year` en el **apartado 2**.

1. **Cuántas** películas están listadas en `movies`.
2. **Cuáles** son las **más antiguas** (menor año extraído del título).
3. **Cuántas** tienen **"Dracula"** en el título (coincidencia parcial, sin distinguir mayúsculas).
4. **Títulos más comunes**.
5. Películas con **"Exorcist"** ordenadas de la más antigua a la más moderna.
6. **Cuántas** con año **1950**.
7. **Cuántas** entre **1950 y 1959** inclusive.
8. **Año** de la película con título exacto **`Batman`** (y contraste con otras *Batman*).
9. Listado de películas que tienen como tag "sci-fi" y "adventure"
10. ¿Cuál es la tag más repetida?

In [ ]:
# 1- Cúantas peliculas están listadas en movies:
print(len(movies_df["movieId"]))
movies_df.shape[0]

In [ ]:
# 2- Cuáles son las más antiguas (menor año extraído del título)
movies_df[movies_df.year == movies_df.year.min()]

In [ ]:
# 3 Cuántas tiene "Dracula" en el titulo
peliculas_dracula = movies_df['title'].str.contains('Dracula', case=False, na=False).sum()
print(f"Cuántas películas contienen Dracula: {peliculas_dracula}")

In [ ]:
# 4- Títulos más comunes
movies_df['title'].value_counts().head(3).reset_index()

In [ ]:
# 4- Títulos más comunes, otra forma
movies_df['title'].value_counts().nlargest(3)

In [ ]:
# 5- Películas con "Exorcist" ordenadas de la más antigua a la más moderna.
movies_df.loc[movies_df.title.str.contains("Exorcist", case=False, na=False), :].sort_values('year')

In [ ]:
# 6- Cuántas con año 1950
print(f"Cantidad de películas que son del año 1950: {(movies_df["year"]==1950).sum()}")
movies_df[movies_df.year == 1950].shape[0]

In [ ]:
# 7- Cuántas entre 1950 y 1959 inclusive
print(f"Cuántas películas hay entre 1950 y 1959: {((movies_df['year'] >= 1950) & (movies_df['year'] <= 1959)).sum()}")
print(f"Cuántas películas hay entre 1950 y 1959: {int(movies_df.year.between(1950, 1959).sum())}")

In [ ]:
# 8- Año de la pelicula con titulo exacto Batam
movies_df[movies_df.title == "Batman"][['title', 'year']]

In [ ]:
# 9- Listado de peliculas que tienen como tag "sci-fi" y "adventure"

# Normalizar texto
tags_movies_df['tag'] = tags_movies_df['tag'].str.lower().str.strip()

# Filtrar tags
tags_buscadas_df = tags_movies_df[tags_movies_df['tag'].isin(['sci-fi', 'adventure'])]

# Contar tags distintos por película
tags_contadas_df = tags_buscadas_df.groupby(['movieId', 'title'], as_index=False).agg(n_tags=('tag', 'nunique'))

# Películas que tienen ambos tags
resultado = tags_contadas_df[tags_contadas_df['n_tags'] == 2]['title'].to_list()

print(f'Hay {len(resultado)} películas etiquetadas como "sci-fi" y "adventure" a la vez.')

In [ ]:
# 10 ¿ Cual es la tag más repetida?
tags_movies_df['tag'].value_counts().head(1)

Una vez terminada la exploración se tiene que arreglar el DataFrame con el que seguir trabajando. Para eso arreglaremos primero `tags`, luego `ratings`, y luego uniremos.

A partir de aquí el DataFrame que cuenta es `movies_unido_df`

In [ ]:
# Normalización de las cadenas de texto de 'tag'
tags_df['tag'] = tags_df['tag'].str.lower().str.strip()

# Agrupación de 'tag' como la columna 'genre' de movies_df
tags_pelicula_df = (tags_df.dropna(subset='tag')
                    .groupby('movieId', as_index=False)['tag'] # agrupa en una misma fila los id y fíjate en la columna tag
                    .apply(lambda tags: '|'.join(tag.capitalize() for tag in sorted(tags.unique())))
                    .reset_index()
                    )

# filtrado de las columnas que nos interesan
tags_pelicula_df = tags_pelicula_df[['movieId', 'tag']]
tags_pelicula_df

In [ ]:
# En el DataFrame de ratings_df nos interesa tener la puntuación por película. Elegimos la media
rating_pelicula_df = ratings_df.groupby('movieId', as_index=False)['rating'].mean().round(2)
rating_pelicula_df

In [ ]:
# Union de todas las tablas
movies_unido_df = (
    movies_df
    .merge(rating_pelicula_df, on='movieId', how='left')
    .merge(tags_pelicula_df, on='movieId', how='left')
    .merge(links_df, on='movieId', how='left')
)

movies_unido_df

In [ ]:
# Corrección numérico columna tmdbId
movies_unido_df['tmdbId'] = pd.to_numeric(movies_unido_df['tmdbId']).astype('Int64')

In [ ]:
# DataFrame con el que seguir el ejercicio
movies_unido_df

## Parte 2: Petición HTTP a la API de TMDB

### Endpoint

`GET https://api.themoviedb.org/3/movie/{tmdb_id}`

Ejemplo público de la misma forma que en la documentación de TMDB: **`https://api.themoviedb.org/3/movie/100`** (el número es el `tmdb_id`; en vuestro caso usaréis el `tmdbId` de cada fila de `links`).

### Tareas

1. Construir un **`DataFrame` `movies10`** con **10 películas** del dataset original (por ejemplo las 10 primeras filas que tengan `tmdbId` tras unir `movies` con `links`).
2. Escribir una función **`fetch_movie_details(tmdb_id)`** que haga la petición anterior y devuelva al menos **`overview`** y **`homepage`** (texto vacío si vienen nulos).
3. Recorrer `movies10` y **añadir** a cada registro esas dos columnas en el propio `DataFrame`.

### Autenticación

Para poder acceder a la API de TMDB, debéis hacer uso del ACCESS TOKEN o de la API KEY que te proporcionan al registrarse. Recomendamos probar los endpoint en POSTMAN para hacer las pruebas de la llamada a la API antes de crear el script en Python. Deberíais tener en vuestro proyecto:**`TMDB_API_KEY`** (query `api_key`) o **`TMDB_READ_ACCESS_TOKEN`** (cabecera `Authorization: Bearer …`). Podéis crear un proyecto con variables de entorno o usar una celda de código usando **`getpass`** (entrada oculta, solo esa sesión del kernel). 

No guardéis claves en el notebook

In [ ]:
# --- Parte 2: TMDB GET /movie/{id} → overview y homepage  ---
# Requiere: `movies` y `links` cargados. pip install requests
import requests
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()  # Carga las variables de entorno desde el archivo .env

# Función para obtener detalles de la película desde TMDB
def fetch_movie_details(tmdb_id: int) -> dict:
    try:
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        # Intentar usar el token de acceso primero, si no está disponible, usar la clave de API
        tmdb_token = os.getenv("TMDB_READ_ACCESS_TOKEN")
        if tmdb_token:
            headers = {"Authorization": f"Bearer {tmdb_token}"}
            response = requests.get(url, headers=headers)
        else:
            response = requests.get(url, params={"api_key": os.getenv("TMDB_API_KEY")}, timeout=10)
        response.raise_for_status()
        return response.json()
    # Manejo de errores
    except requests.HTTPError as e:
        print(f"HTTP error for ID {tmdb_id}: {e}")
        return {}
    except requests.Timeout as e:
        print(f"Request timed out for ID {tmdb_id}: {e}")
        return {}
    except ValueError as e:
        print(f"Error parsing JSON for ID {tmdb_id}: {e}")
        return {}
    except requests.RequestException as e:
        print(f"Error fetching movie details for ID {tmdb_id}: {e}")
        return {}
    except Exception as e:
        print(f"Unexpected error for ID {tmdb_id}: {e}")
        return {}

# Incorporamos las columnas `overview` y `homepage` al DataFrame de películas
movies10 = movies_df.head(10).copy().merge(links_df, on='movieId', how='left').head(10)
movies10["tmdbId"] = round(movies10["tmdbId"].astype(float), 0).astype('Int64')  # Convertimos a Int64 para permitir nulos

# Iteramos sobre las primeras 10 películas y obtenemos sus detalles de TMDB
for idx, row in movies10.iterrows():
    # Solo procesamos si `tmdbId` no es nulo
    if pd.isna(row["tmdbId"]):
        continue
    details = fetch_movie_details(row["tmdbId"])
    # Si no se obtienen detalles, imprimimos un mensaje y continuamos con la siguiente película
    if not details:
        print(f"No details found for movie ID {row['tmdbId']}")
        continue
    else:
        # Guardamos los detalles obtenidos en las columnas correspondientes
        # Si no se encuentra la clave, se asigna una cadena vacía
        print(f"Fetched details for movie ID {row['tmdbId']}")
        movies10.at[idx, "overview"] = details.get("overview", "")
        movies10.at[idx, "homepage"] = details.get("homepage", "")



## Parte 3: Sinopsis en español con Gemini

Usad el `DataFrame` **`movies10`** de la Parte 2 (columna `overview` en inglés desde TMDB).

### Tareas

1. Configurar **`GEMINI_API_KEY`** (variable de entorno o `getpass`, como en Sprint 4).
2. Crear el cliente **`google-genai`** y una función **`summarize_overview_es(overview, title="")`** que devuelva un **resumen en español de máximo 2 frases**. Si `overview` está vacío, devolver cadena vacía **sin llamar a la API**.
3. Recorrer `movies10` y añadir la columna **`overview_es`**.
4. Mostrar `title`, `overview` (recorte) y `overview_es` para **3 películas**.

### Autenticación

Clave en [Google AI Studio](https://aistudio.google.com/). Variable **`GEMINI_API_KEY`** o celda con **`getpass`**. No guardéis claves en el notebook.

### Prerrequisitos

- Parte 2 ejecutada (`movies10` con columna `overview`).
- `pip install google-genai`

In [ ]:
import os, getpass 
import pandas as pd
import requests
from dotenv import load_dotenv
from google import genai
# PRE-REQUISITOS: de parte 2 → → → columna 'overview' y aunque no lo diga, también debería venir la columna 'title' 
# asumiendo lo anterior...

# carga .env para los secretos de entorno (.environment-secrets) y establece la API KEY según lo recuperado 
# desde el operativo en el archivo de secretos de entorno (.env) para su uso con 'genai.Client()'
load_dotenv()
cliente = genai.Client(api_key = os.getenv("GEMINI_API_KEY"))
# si no hay API en .env solicita su entrada de manera velada por teclado (lógicamente vale 'CTRL + V' para pegarla...)
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Introduce GEMINI_API_KEY: ")

# Carga de .env el modelo por defecto o, en caso de no conseguirlo, asume "gemini-3.5-flash"
MODEL = os.getenv("GEMINI_MODEL") or "gemini-3.5-flash"
# genero un .env.examples con esas dos variables (solo una a mayores)
# BIEN: 
# Garantizados movies10 (parte 2 anterior) y google-genai instalado por (%pip entorno virtual)
# que aportan el campo 'overview'y 'title' ambos de tipo string:
# --------------------------------------------------------------
# NOTA: el modelo toma el valor de la variable de entorno 'GEMINI_MODEL' de .env.
#       He generado un .env.example con la variable GEMINI_MODEL = 'gemini-3.5-flash' además de la GEMINI_API_KEY = 'paste_your_api_key_here_between_quotes'
def summarize_overview_es(overview: str, title="") -> str:    
    if not overview:
        return ""
    
    prompt = ("Summarise the following text in Spanish, in no more than two sentences, without altering its original" + 
              "meaning and always preserving its resemblance to the original language: " + overview
              )
    
    respuesta = cliente.models.generate_content(
        model = MODEL,
        contents = prompt
    )
    return respuesta.text

# Asumiendo movies10[overview, title] como inputs de la función a ejecutar que grabará la nueva columna:
# ------------------------------------------------------------------------------------------------------
# Aplica la función a cada fila usando la columna overview_es
# Utilizamos lambda como pequeño ayudante a la hora de aplicar la función a cada fila de la nueva columna overview_es, 
# que es la misma salida de la función de llamada al agente de Gemini que resume en español (según enunciado).
movies10["overview_es"] = movies10.apply(
    lambda row: summarize_overview_es(
        overview=row["overview"],
        title=row.get("title", "")
    ),
    axis=1
)

## Parte 4 (opcional): extensiones con Gemini

**No obligatorio.** Solo si el grupo terminó las partes 1-3. Podéis elegir **A**, **B** o ambas. Son independientes.

---

### A) Catálogo ampliado en `movies10`

Partiendo de `movies10` (con `overview` y, si ya lo tenéis, `overview_es`), añadid con **una llamada JSON por película**:

- **`pitch_es`**: texto de cartelera en español (máx. 280 caracteres).
- **`edad_sugerida`**: uno de `TP`, `+7`, `+12`, `+16`, `+18`.
- **`temas`**: exactamente 3 temas en español (en el DataFrame, cadena separada por comas).

Función sugerida: **`enrich_catalog_fields(overview, title="", genres="", year=None)`**. Basad la respuesta solo en sinopsis y metadatos; no inventéis reparto ni datos externos. Si `overview` está vacío, no llaméis a la API.

Mostrad 2–3 filas con las columnas nuevas.

---

### B) Recomendador por género

Usad **`ratings_movies`** (Parte 1) y Gemini.

1. **`top_rated_by_genre(ratings_movies, genres, top_n=10, min_ratings=50)`** — filtra por género(s), nota media por `movieId`, devuelve el top N (mínimo `min_ratings` valoraciones por película).
2. **`recommend_movies(candidates, favorite_genres, n=3)`** — el modelo recomienda **solo** del catálogo candidato, con una frase de justificación en español cada una.
3. Probad con 1–2 géneros (p. ej. `["Action", "Sci-Fi"]`): mostrad candidatas y respuesta del modelo.

In [ ]:
# --- Parte 4A (opcional): pitch_es, edad_sugerida, temas ---
# Requiere: `movies10` con `overview`; `client` y `MODEL` de la Parte 3.

In [ ]:
# Definicion de la funcion enrich_catalog_fields(overview, title="", genres="", year=None)
def enrich_catalog_fields(overview: str, title: str = "", genre: str = "", , year=None) -> dict:
    """
    Amplia campos utilizando Gemini.

    Si overview está vacío no llama a la API.

    Devuelve:
    - pitch_es: texto de cartelera en español (280 caracteres)
    - edad_sugerida: TP, +7, +12, +16, +18
    - temas: 3 temas en español separados por coma.
    """

    # Si no hay overview devuelve diccionario sin valores.
    if pd.isna(overview) or overview == "":
        return {
            "pitch_es": "",
            "edad_sugerida": "",
            "temas": "" 
        }
    
    prompt = f"""
Eres un generador de información para un catálogo de películas.
Basándote únicamente en la sinopsis y los metadatos proporcionados genera un JSON válido.
No inventes reparto, directores, datos externos, ni información que no se encuentre en los datos proporcionados.

Datos de la película:
- Título: {title}
- Género: {genre}
- Año: {year}
- Sinopsis: {overview}

Devuleve un JSON con la siguiente estructura:

{{
    "pitch_es": texto breve de cartelera en español, máximo 280 caracteres,
    "edad_sugerida": una entre: TP, +7, +12, +16, +18,
    "temas": ["tema 1", "tema 2", "tema 3"]
}}

Reglas:
- pitch_es tiene que estar en español y no exceder los 280 caracteres. 
- edad_sugerida solo puede tomar uno de los valores propuestos. 
- temas debe contener exactamente 3 temas.
- no puede haber texto ni antes ni después del JSON.
    """
    load_dotenv()

    # carga de modelo
    MODEL=os.gentenv("GEMINI_MODEL") or "gemini-3.5-flash"

    # carga de la API_KEY
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

    if not GEMINI_API_KEY:
        os.environ["GEMINI_API_KEY"] = getpass.getpass("Introduce GEMINI_API_KEY: ")

    # genera cliente
    cliente = genai.Client(api_key=GEMINI_API_KEY)


    try:
        respuesta = cliente.models.generate_content(
            model=MODEL
            contents=prompt
        )

        # Se quitan espacios de principio y fin
        texto = respuesta.strip()

        # Si hay un ```json inicial se quita
        texto = texto.replace("```json", "").replace("```", "").strip()

        # parseo a JSON
        data = json.loads(texto)

        # información requerida por enunciado
        pitch_es = data.get("pitch_es", "")
        edad_sugerida = data.get("edad_sugerida", "")
        temas = data.get("temas", [])

        # validacion de pitch_es
        if len(pitch_es) > 280:
            pitch_es = pitch_es[:280]
        
        # validacion de edad_sugerida
        edades_validas = ["TP", "+7", "+12", "+16", "+18"]

        if edad_sugerida not in edades_validas:
            edad_sugerida = ""

        # validacion de temas
        if isinstance(temas, list):
            temas = temas[:3]

            while len(temas) < 3:
                temas.append("")

            temas = ", ".join(temas)
        else:
            temas = ""
        
        return {
                "pitch_es": pitch_es,
                "edad_sugerida": edad_sugerida,
                "temas": temas
            }

    except Exception as e:
        print(f"Error enriqueciendo '{title}': {e}")

        return {
            "pitch_es": "",
            "edad_sugerida": "",
            "temas": ""
        }


In [ ]:
# Aplicar la función enrich_catalog_fields
catalog_data = movies10.apply(
    lambda row: enrich_catalog_fields(
        overview=row["overview"],
        title=row.get("title", ""),
        genre=row.get("genre", ""),
        year=row.get("year", None)
    ),
    axis=1
)

# conversión a DataFrame
catalog_df = pd.DataFrame(catalog_data.tolist())

# añadido de nuevas columnas a movies10
movies10["pitch_es"] = catalog_df["pitch_es"]
movies10["edad_sugerida"] = catalog_df["edad_sugerida"]
movies10["temas"] = catalog_df["temas"]

# mostrar las primeras filas
movies10[
    [
        "title",
        "overview_es",
        "pitch_es",
        "edad_sugerida",
        "temas"
    ]
].head(3)

In [ ]:
# --- Parte 4B (opcional): recomendador por género (SOLUCIÓN) ---
# Requiere: `ratings_movies` (Parte 1); `client` y `MODEL` (Parte 3).